# Milestone 4 — Collapsed Gibbs vs Heuristic Sampler

Compares two samplers on the 18 test homes:

- **Old** (`infer_all`): z sampled from C=0/C=1 mixture via FFBS; C re-sampled from logistic regression on transitions-per-day.
- **New** (`infer_all_collapsed`): C sampled first from its exact marginal posterior $p(C \mid x, \alpha, \Theta) \propto p(C)\,p(x \mid C, \alpha, \Theta)$, with the HMM marginal likelihood $p(x \mid C{=}1)$ coming from the forward pass. z drawn conditionally on C.

Both return `HomeInference` objects with identical structure, so `evaluate()` works on both.

In [8]:
import sys
from pathlib import Path

repo_root = str(Path('.').resolve().parents[2])
utils_dir = str(Path('.').resolve().parents[2] / 'notebooks' / 'utils')
for p in [repo_root, utils_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from models import graphical_model as gm
from models import first_diff_logistic as fdl
import convergence_plots as cvg
from data_utils import load_split

## 1. Fitted parameters

In [ ]:
params_path = Path('../../../models/fitted_params.pkl')
if params_path.exists():
    params = pickle.load(open(params_path, 'rb'))
    print('Loaded pre-fitted params.')
else:
    print('fitted_params.pkl not found — fitting on train set...')
    train_df = load_split('../../../data_processing/splits/train.pkl')
    train_df['non_ev_load'] = train_df['total_load'] - train_df['ev_load']
    params = gm.fit(train_df, verbose=True)
    pickle.dump(params, open(params_path, 'wb'))
    print(f'Saved to {params_path.resolve()}')

print(params.summary())

## 2. Test set + heuristic warm-start

In [ ]:
test_df = load_split('../../../data_processing/splits/test.pkl')
test_df['non_ev_load'] = test_df['total_load'] - test_df['ev_load']

n_test = test_df.home_id.nunique()
n_ev   = test_df.groupby('home_id')['has_ev'].first().sum()
print(f'Test: {n_test} homes  ({n_ev} EV, {n_test - n_ev} non-EV)')

# Fit heuristic on train (warm-start only — inference itself uses total_load alone)
train_df = load_split('../../../data_processing/splits/train.pkl')
homes_train = gm.build_heuristic_homes(train_df)
logistic_model, w, lo, hi, md = fdl.tune(homes_train)

homes_test = gm.build_heuristic_homes(test_df)
heuristic_summary, heuristic_states = fdl.predict(logistic_model, homes_test, w, lo, hi, md)
heuristic_summary['C_hat'] = (heuristic_summary['p_hat'] >= 0.5).astype(int)

init_c_dict = dict(zip(heuristic_summary['dataid'].astype(int),
                        heuristic_summary['C_hat'].astype(int)))
init_z_dict = {}
for hid, z_flat in heuristic_states.items():
    g = test_df[test_df['home_id'] == hid]
    D = g['day'].nunique()
    init_z_dict[int(hid)] = z_flat[:D * 96].reshape(D, 96)

print('\nHeuristic predictions:')
print(heuristic_summary.to_string(index=False))

## 3. Sampler config

In [ ]:
S_BURN = 200
S      = 500
SEED   = 42

## 4. Marginal likelihood diagnostic — log_Z1 vs log_Z0

Before running any Gibbs iterations, compute $\log p(x \mid C=1, \alpha, \Theta)$ (the HMM forward-pass marginal) and $\log p(x \mid C=0, \alpha)$ (the all-off likelihood) for every test home at the prior means $\alpha = \mu_\alpha$, $\Theta = \mu_\Theta$.

These are the two quantities the collapsed C sampler compares at each iteration. The plots show their raw scale and whether the gap is meaningful or overwhelmed by model flexibility.

In [ ]:
log_pi = np.log(params.pi_z + 1e-300)
log_P  = np.log(params.P_z  + 1e-300)

diag_rows = []
for hid in sorted(test_df['home_id'].unique()):
    g     = test_df[test_df['home_id'] == hid].sort_values(['day', 'time_index'])
    D     = g['day'].nunique()
    x     = g['total_load'].to_numpy().reshape(D, gm.T).astype(np.float64)
    tc    = int(g['has_ev'].iloc[0])

    _, log_Z1 = gm._hmm_forward(x, params.mu_theta.copy(), params.mu_alpha,
                                 params, log_pi, log_P)
    log_Z0    = gm._compute_loglik_c0(x, params.mu_alpha, params)

    log_prior_odds = np.log(params.p_C + 1e-300) - np.log(1 - params.p_C + 1e-300)
    gap            = log_Z1 - log_Z0
    p_c1           = float(1 / (1 + np.exp(-(log_prior_odds + gap))))

    diag_rows.append(dict(home_id=hid, true_C=tc, D=D,
                          log_Z0=log_Z0, log_Z1=log_Z1,
                          gap=gap, gap_per_step=gap / (D * gm.T),
                          log_prior_odds=log_prior_odds, p_c1=p_c1))

ddf = pd.DataFrame(diag_rows).sort_values(['true_C', 'gap'])

print(ddf[['home_id','true_C','D','log_Z0','log_Z1','gap','gap_per_step','p_c1']]
      .to_string(index=False, float_format='{:.1f}'.format))

# ── plot ──────────────────────────────────────────────────────────────────────
COLORS = {0: 'steelblue', 1: 'tomato'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: scatter log_Z0 vs log_Z1 — points above y=x mean HMM wins
ax = axes[0]
for tc, grp in ddf.groupby('true_C'):
    ax.scatter(grp['log_Z0'], grp['log_Z1'],
               color=COLORS[tc], s=70, zorder=3,
               label='EV' if tc else 'non-EV')
lims = [min(ddf['log_Z0'].min(), ddf['log_Z1'].min()),
        max(ddf['log_Z0'].max(), ddf['log_Z1'].max())]
ax.plot(lims, lims, 'k--', lw=1, label='y = x  (gap = 0)')
ax.set_xlabel('log p(x | C=0, α)  —  log_Z0')
ax.set_ylabel('log p(x | C=1, α, Θ)  —  log_Z1')
ax.set_title('log_Z1 vs log_Z0\nAbove dashed line → HMM wins')
ax.legend()

# Panel 2: total gap per home, with the prior log-odds threshold marked
ax = axes[1]
xpos = np.arange(len(ddf))
bars = ax.bar(xpos, ddf['gap'],
              color=[COLORS[tc] for tc in ddf['true_C']], alpha=0.85)
for bar, tc in zip(bars, ddf['true_C']):
    if tc == 1:
        bar.set_edgecolor('k'); bar.set_linewidth(1.5)
threshold = -ddf['log_prior_odds'].iloc[0]
ax.axhline(0, color='k', lw=0.8)
ax.axhline(threshold, color='purple', lw=1.5, ls='--',
           label=f'−log prior odds = {threshold:.1f} nats\n(gap above this → P̂(C=1) > 0.5)')
ax.set_xticks(xpos)
ax.set_xticklabels(ddf['home_id'].astype(str), rotation=45, ha='right', fontsize=8)
ax.set_ylabel('log_Z1 − log_Z0  (nats)')
ax.set_title('Total likelihood gap\nover all D×T observations')
ax.legend(fontsize=8)

# Panel 3: gap normalised per timestep — shows per-observation scale
ax = axes[2]
bars = ax.bar(xpos, ddf['gap_per_step'],
              color=[COLORS[tc] for tc in ddf['true_C']], alpha=0.85)
for bar, tc in zip(bars, ddf['true_C']):
    if tc == 1:
        bar.set_edgecolor('k'); bar.set_linewidth(1.5)
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(xpos)
ax.set_xticklabels(ddf['home_id'].astype(str), rotation=45, ha='right', fontsize=8)
ax.set_ylabel('(log_Z1 − log_Z0) / (D·T)  (nats per step)')
ax.set_title('Gap per observation\n(tiny per step, but × D·T ≈ 34k)')

from matplotlib.patches import Patch
fig.legend(handles=[Patch(color='tomato', label='EV (outlined bar)'),
                    Patch(color='steelblue', label='non-EV')],
           loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.02))
plt.suptitle('Evaluated at prior means α=μ_α, Θ=μ_Θ — before any Gibbs updates',
             y=1.06, fontsize=10)
plt.tight_layout()
plt.show()

## 4. Old sampler — logistic heuristic C step

In [ ]:
inferences_old = gm.infer_all(
    test_df, params,
    S_burn=S_BURN, S=S, seed=SEED,
    verbose=True,
    initial_c_dict=init_c_dict,
    initial_z_dict=init_z_dict,
    c_logistic_model=logistic_model,
)

## 5. Collapsed Gibbs — exact marginal posterior C step

Reload the module first so any code changes on disk are picked up without restarting the kernel.

In [ ]:
import importlib, models.graphical_model as _gm_mod
importlib.reload(_gm_mod)
import models.graphical_model as gm   # rebind to the fresh version

inferences_new = gm.infer_all_collapsed(
    test_df, params,
    S_burn=S_BURN, S=S, seed=SEED,
    verbose=True,
)

## 5b. Marginal likelihood traces during the chain

For each home, `log_Z1` and `log_Z0` are computed fresh at every Gibbs iteration (with updated α and Θ). Plotting them over iterations answers:
- Does the gap grow during burn-in? (feedback loop: Θ adapts to noise → HMM wins more)
- Is the gap stable from iteration 1? (structural issue, not a chain dynamics problem)
- Is there any difference in gap trajectory between EV and non-EV homes?

In [ ]:
true_c     = test_df.groupby('home_id')['has_ev'].first().to_dict()
ev_hids    = [hid for hid in sorted(inferences_new) if true_c[hid]]
nonev_hids = [hid for hid in sorted(inferences_new) if not true_c[hid]]

bad_nonev  = sorted(nonev_hids, key=lambda h: ddf.loc[ddf.home_id==h, 'gap'].values[0], reverse=True)[:3]
good_nonev = sorted(nonev_hids, key=lambda h: ddf.loc[ddf.home_id==h, 'gap'].values[0])[:2]

def get_traces(inf):
    """Return (log_Z1_trace, log_Z0_trace) or (None, None) if not recorded."""
    return (getattr(inf, 'log_Z1_trace', None),
            getattr(inf, 'log_Z0_trace', None))

# Check whether traces are available at all
sample_z1, _ = get_traces(inferences_new[ev_hids[0]])
if sample_z1 is None:
    print("log_Z1_trace not found — re-run cell 5 (collapsed Gibbs) to record traces.")
else:
    plot_groups = {
        'EV':            ev_hids,
        'non-EV (bad)':  bad_nonev,
        'non-EV (good)': good_nonev,
    }

    fig, axes = plt.subplots(len(plot_groups), 1, figsize=(14, 4 * len(plot_groups)), sharex=False)
    for ax, (group_label, hids) in zip(axes, plot_groups.items()):
        for hid in hids:
            z1, z0 = get_traces(inferences_new[hid])
            if z1 is None:
                continue
            gap   = z1 - z0
            color = 'tomato' if true_c[hid] else 'steelblue'
            ax.plot(gap, lw=0.8, alpha=0.85, color=color,
                    label=f'home {hid} ({"EV" if true_c[hid] else "non-EV"})')
        ax.axvline(S_BURN, color='k', lw=1, ls='--', alpha=0.5, label=f'burn-in end ({S_BURN})')
        ax.axhline(0, color='k', lw=0.5)
        ax.set_ylabel('log_Z1 − log_Z0  (nats)')
        ax.set_title(group_label)
        ax.legend(fontsize=8, loc='upper right')
        ax.set_xlabel('Gibbs iteration')
    plt.suptitle('Marginal likelihood gap over Gibbs iterations\n'
                 'Computed fresh each step with current α and Θ', fontsize=11)
    plt.tight_layout()
    plt.show()

    # Raw traces for one EV and one bad non-EV
    fig, axes = plt.subplots(2, 2, figsize=(14, 7))
    for col, (hid, label) in enumerate([(ev_hids[0], 'EV'), (bad_nonev[0], 'non-EV (worst gap)')]):
        z1, z0 = get_traces(inferences_new[hid])
        iters  = np.arange(len(z1))

        ax = axes[0, col]
        ax.plot(iters, z1, lw=0.7, color='tomato',    label='log_Z1  (HMM)')
        ax.plot(iters, z0, lw=0.7, color='steelblue', label='log_Z0  (all-off)')
        ax.axvline(S_BURN, color='k', lw=1, ls='--', alpha=0.5)
        ax.set_title(f'Home {hid} ({label}) — raw log-likelihoods')
        ax.set_ylabel('log p  (nats)')
        ax.legend(fontsize=8)

        ax = axes[1, col]
        ax.plot(iters, z1 - z0, lw=0.7, color='purple')
        ax.axvline(S_BURN, color='k', lw=1, ls='--', alpha=0.5)
        ax.axhline(0, color='k', lw=0.5)
        ax.set_title(f'Home {hid} — gap (log_Z1 − log_Z0)')
        ax.set_xlabel('Gibbs iteration')
        ax.set_ylabel('nats')

    plt.suptitle('Are log_Z1 and log_Z0 stable or drifting as α and Θ update?', fontsize=11)
    plt.tight_layout()
    plt.show()

## 6. C classification comparison

In [ ]:
true_c = test_df.groupby('home_id')['has_ev'].first().to_dict()

rows = []
for hid in sorted(inferences_old.keys()):
    tc  = int(true_c[hid])
    hp  = float(heuristic_summary.loc[heuristic_summary['dataid'] == hid, 'p_hat'].iloc[0])
    op  = float(inferences_old[hid].c_samples.mean())
    np_ = float(inferences_new[hid].c_samples.mean())
    rows.append(dict(home_id=hid, true_C=tc,
                     heur_p=hp,  heur_hat=int(hp  >= 0.5),
                     old_p=op,   old_hat=int(op   >= 0.5),
                     new_p=np_,  new_hat=int(np_  >= 0.5)))

cmp = pd.DataFrame(rows)

for label, col in [('Heuristic', 'heur_hat'), ('Old Gibbs', 'old_hat'), ('Collapsed Gibbs', 'new_hat')]:
    acc = (cmp[col] == cmp['true_C']).mean()
    print(f'{label:>20}: accuracy = {acc:.3f}')

print()
display(cmp.style
    .format({'heur_p': '{:.3f}', 'old_p': '{:.3f}', 'new_p': '{:.3f}'})
    .apply(lambda col: [
        'background-color: #d4edda' if v == cmp.loc[i, 'true_C']
        else 'background-color: #f8d7da'
        for i, v in col.items()
    ], subset=['heur_hat', 'old_hat', 'new_hat'])
    .set_caption('Green = correct, red = wrong')
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for ax, (title, pcol, hatcol) in zip(axes, [
    ('Heuristic',             'heur_p', 'heur_hat'),
    ('Old Gibbs (logistic C)', 'old_p',  'old_hat'),
    ('Collapsed Gibbs',        'new_p',  'new_hat'),
]):
    ev_mask = cmp['true_C'] == 1
    acc = (cmp[hatcol] == cmp['true_C']).mean()
    for mask, color, label in [(~ev_mask, 'steelblue', 'non-EV'), (ev_mask, 'tomato', 'EV')]:
        ax.barh(cmp.loc[mask, 'home_id'].astype(str), cmp.loc[mask, pcol],
                color=color, label=label, alpha=0.85)
    ax.axvline(0.5, color='k', lw=1, ls='--')
    ax.set_xlim(0, 1)
    ax.set_xlabel('P̂(C=1)')
    ax.set_title(f'{title}\nacc = {acc:.3f}')

axes[0].legend(loc='lower right')
plt.suptitle('P̂(C=1) per test home   |   red = EV, blue = non-EV, dashed = decision boundary')
plt.tight_layout()
plt.show()

## 7. Full evaluation (z + C confusion matrices)

In [ ]:
print('=== OLD SAMPLER ===')
results_old = gm.evaluate(
    test_df, inferences_old,
    c_prob_methods={
        'old_gibbs': {hid: float(inf.c_samples.mean()) for hid, inf in inferences_old.items()},
        'heuristic': dict(zip(heuristic_summary['dataid'].astype(int),
                              heuristic_summary['p_hat'].astype(float))),
    },
    heuristic_states=heuristic_states,
)
gm.print_evaluation(results_old)

In [ ]:
print('=== COLLAPSED GIBBS ===')
results_new = gm.evaluate(
    test_df, inferences_new,
    c_prob_methods={
        'collapsed': {hid: float(inf.c_samples.mean()) for hid, inf in inferences_new.items()},
        'heuristic': dict(zip(heuristic_summary['dataid'].astype(int),
                              heuristic_summary['p_hat'].astype(float))),
    },
    heuristic_states=heuristic_states,
)
gm.print_evaluation(results_new)

## 8. Posterior plots — EV homes

In [ ]:
ev_hids = [hid for hid in sorted(inferences_old)
           if bool(test_df[test_df['home_id'] == hid]['has_ev'].iloc[0])]

for hid in ev_hids:
    inf_o = inferences_old[hid]
    inf_n = inferences_new[hid]
    g = test_df[test_df['home_id'] == hid].sort_values(['day', 'time_index'])
    D = g['day'].nunique()
    z_true = g['charge_state'].to_numpy().reshape(D, 96)

    fig = plt.figure(figsize=(16, 11))
    fig.suptitle(f'Home {hid}  (EV)  |  '
                 f'old P̂(C=1) = {inf_o.c_samples.mean():.3f}  '
                 f'collapsed P̂(C=1) = {inf_n.c_samples.mean():.3f}', fontsize=12)
    gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.35)

    # C chain
    ax = fig.add_subplot(gs[0, :2])
    ax.plot(inf_o.c_samples, lw=0.6, alpha=0.8, label='old')
    ax.plot(inf_n.c_samples, lw=0.6, alpha=0.8, label='collapsed')
    ax.set_title('C samples (post burn-in)'); ax.set_xlabel('retained sample')
    ax.set_ylabel('C'); ax.legend()

    # α
    ax = fig.add_subplot(gs[0, 2:])
    ax.hist(inf_o.alpha_samples, bins=40, alpha=0.6, label='old', density=True)
    ax.hist(inf_n.alpha_samples, bins=40, alpha=0.6, label='collapsed', density=True)
    ax.set_title('α posterior'); ax.set_xlabel('α (kW)'); ax.legend()

    # Θ low / high
    for ki, (k, name) in enumerate([(1, 'low'), (2, 'high')]):
        ax = fig.add_subplot(gs[1, ki*2:(ki+1)*2])
        ax.hist(inf_o.theta_samples[:, k], bins=40, alpha=0.6, label='old', density=True)
        ax.hist(inf_n.theta_samples[:, k], bins=40, alpha=0.6, label='collapsed', density=True)
        ax.axvline(params.mu_theta[k], color='k', ls='--', lw=1, label='prior mean')
        ax.set_title(f'Θ_{name} posterior'); ax.set_xlabel('kW'); ax.legend()

    # z heatmaps
    for col, (z_plot, label) in enumerate([
        (z_true,       'Ground truth z'),
        (inf_o.z_hat,  'Old Gibbs MAP z'),
        (inf_n.z_hat,  'Collapsed MAP z'),
    ]):
        ax = fig.add_subplot(gs[2, col])
        ax.imshow(z_plot.T, aspect='auto', cmap='viridis', vmin=0, vmax=2,
                  interpolation='nearest')
        ax.set_title(label); ax.set_xlabel('day'); ax.set_ylabel('time')

    ax = fig.add_subplot(gs[2, 3])
    p_chg = 1.0 - inf_n.z_marginals[:, :, 0]
    im = ax.imshow(p_chg.T, aspect='auto', cmap='Reds', vmin=0, vmax=1,
                   interpolation='nearest')
    ax.set_title('Collapsed P(z≠off)'); ax.set_xlabel('day')
    plt.colorbar(im, ax=ax, fraction=0.046)

    plt.show()

## 9. Posterior plots — non-EV homes with highest P̂(C=1)

In [ ]:
nonev_hids = [hid for hid in sorted(inferences_old)
              if not bool(test_df[test_df['home_id'] == hid]['has_ev'].iloc[0])]
plot_nonev = sorted(nonev_hids,
    key=lambda h: inferences_old[h].c_samples.mean(), reverse=True)[:4]

for hid in plot_nonev:
    inf_o = inferences_old[hid]
    inf_n = inferences_new[hid]

    fig, axes = plt.subplots(1, 3, figsize=(14, 3))
    fig.suptitle(f'Home {hid}  (non-EV)  |  '
                 f'old P̂(C=1) = {inf_o.c_samples.mean():.3f}  '
                 f'collapsed P̂(C=1) = {inf_n.c_samples.mean():.3f}', fontsize=11)

    axes[0].plot(inf_o.c_samples, lw=0.5, alpha=0.7, label='old')
    axes[0].plot(inf_n.c_samples, lw=0.5, alpha=0.7, label='collapsed')
    axes[0].set_title('C samples'); axes[0].set_xlabel('retained sample')
    axes[0].set_ylabel('C'); axes[0].legend()

    axes[1].hist(inf_o.alpha_samples, bins=30, alpha=0.6, label='old', density=True)
    axes[1].hist(inf_n.alpha_samples, bins=30, alpha=0.6, label='collapsed', density=True)
    axes[1].set_title('α posterior'); axes[1].set_xlabel('α (kW)'); axes[1].legend()

    p_chg_o = (1.0 - inf_o.z_marginals[:, :, 0]).mean(axis=0)
    p_chg_n = (1.0 - inf_n.z_marginals[:, :, 0]).mean(axis=0)
    axes[2].plot(p_chg_o, lw=1, label='old')
    axes[2].plot(p_chg_n, lw=1, label='collapsed')
    axes[2].set_title('Mean P(z≠off) by time-of-day')
    axes[2].set_xlabel('15-min interval'); axes[2].legend()

    plt.tight_layout()
    plt.show()

## 10. Convergence diagnostics

In [ ]:
diag_hid = ev_hids[0]
print(f'Diagnostics for home {diag_hid} (EV)')

for label, inf in [('Old sampler', inferences_old[diag_hid]),
                   ('Collapsed Gibbs', inferences_new[diag_hid])]:
    print(f'\n--- {label} ---')
    if inf.alpha_trace is not None:
        figs = cvg.plot_all_diagnostics(inf)
        for fig in figs:
            fig.suptitle(f'{label} — home {diag_hid}', fontsize=11)
            plt.show()
    else:
        print('Traces not recorded.')

## 11. C sample variance by class

Lower variance on non-EV homes = the sampler is more decisive. EV homes should be similar between the two.

In [ ]:
var_rows = []
for hid in sorted(inferences_old):
    tc = int(true_c[hid])
    var_rows.append(dict(
        home_id  = hid,
        true_C   = tc,
        old_p    = float(inferences_old[hid].c_samples.mean()),
        old_varC = float(np.var(inferences_old[hid].c_samples)),
        new_p    = float(inferences_new[hid].c_samples.mean()),
        new_varC = float(np.var(inferences_new[hid].c_samples)),
    ))

vdf = pd.DataFrame(var_rows)
print('Mean Var(C) by true class:')
print(vdf.groupby('true_C')[['old_varC', 'new_varC']].mean().round(4))
print()
display(vdf.style.format({
    'old_p': '{:.3f}', 'old_varC': '{:.4f}',
    'new_p': '{:.3f}', 'new_varC': '{:.4f}',
}))